# Neural networks: layers, XOR, and one gradient update

Companion to HTB Academy Module 290, Section 18. Part 1 uses **fixed, hand-designed weights** to show how nonlinear hidden neurons express XOR. Part 2 traces **one illustrative training step** through a different, tiny two-layer network. Neither is a production security model.

Read hⱼ=f(Σᵢwⱼᵢxᵢ+bⱼ) as **h sub j equals f of the sum of w sub j-i times x sub i, plus b sub j**. Each neuron consumes the previous layer's values, applies weights and bias, then an activation.

In [ ]:
def relu(z):
    """ReLU(z) = max(0, z): the nonlinear hidden activation."""
    return max(0.0, z)

def xor_network(x1, x2):
    """A hand-designed two-hidden-neuron forward pass, not a trained model."""
    h1 = relu(x1 - x2)  # First hidden unit detects (1, 0).
    h2 = relu(x2 - x1)  # Second hidden unit detects (0, 1).
    output = h1 + h2  # For binary inputs, this is exactly XOR's 0 or 1.
    return h1, h2, output

for inputs, expected in [((0, 0), 0), ((1, 0), 1), ((0, 1), 1), ((1, 1), 0)]:
    h1, h2, result = xor_network(*inputs)
    print(f'inputs={inputs} -> hidden=({h1:.0f}, {h2:.0f}) -> output={result:.0f}')
    assert result == expected

## One small chain-rule calculation

Use h=ReLU(w₁x+b₁), prediction ŷ=w₂h+b₂, and loss L=½(ŷ−target)². Read ∂L/∂w₁ as **partial L over partial w sub one**: the local loss slope for the first weight. At positive hidden input, ReLU's derivative is 1. The chain rule multiplies the slopes along the path from w₁ to h to ŷ to L. Backpropagation computes those slopes; gradient descent then updates the parameters.

In [ ]:
x, target = 2.0, 2.0
w1, b1, w2, b2 = 0.5, 0.0, 1.0, 0.0
learning_rate = 0.1  # η, pronounced eta.

def forward_and_loss(w1, b1, w2, b2):
    hidden_input = w1 * x + b1
    hidden = relu(hidden_input)
    prediction = w2 * hidden + b2
    loss = 0.5 * (prediction - target) ** 2
    return hidden_input, hidden, prediction, loss

hidden_input, hidden, prediction, old_loss = forward_and_loss(w1, b1, w2, b2)
assert (hidden_input, hidden, prediction, old_loss) == (1.0, 1.0, 1.0, 0.5)
print('before:', 'hidden=', hidden, 'prediction=', prediction, 'loss=', old_loss)

# Backward pass. Here hidden_input=1>0, so the local ReLU slope is 1.
d_loss_d_prediction = prediction - target  # Derivative of half squared error.
d_relu_d_input = 1.0 if hidden_input > 0 else 0.0
d_loss_d_w2 = d_loss_d_prediction * hidden
d_loss_d_b2 = d_loss_d_prediction
d_loss_d_hidden = d_loss_d_prediction * w2
d_loss_d_w1 = d_loss_d_hidden * d_relu_d_input * x
d_loss_d_b1 = d_loss_d_hidden * d_relu_d_input
print('gradients:', d_loss_d_w1, d_loss_d_b1, d_loss_d_w2, d_loss_d_b2)
assert (d_loss_d_w1, d_loss_d_b1, d_loss_d_w2, d_loss_d_b2) == (-2.0, -1.0, -1.0, -1.0)

# Gradient descent: each new parameter = old parameter - η * its gradient.
new_w1 = w1 - learning_rate * d_loss_d_w1
new_b1 = b1 - learning_rate * d_loss_d_b1
new_w2 = w2 - learning_rate * d_loss_d_w2
new_b2 = b2 - learning_rate * d_loss_d_b2
_, new_hidden, new_prediction, new_loss = forward_and_loss(new_w1, new_b1, new_w2, new_b2)
print('after:', 'hidden=', new_hidden, 'prediction=', new_prediction, 'loss=', new_loss)
assert abs(new_loss - 0.03125) < 1e-12
assert new_loss < old_loss

## Sanity-check one derivative numerically

A centered finite difference estimates the slope by measuring loss a little to either side of w₁. It should be close to −2 here. This is a teaching check, not a replacement for backpropagation in a real network. The ReLU corner at exactly zero needs special derivative conventions, so our example deliberately stays on the positive side.

In [ ]:
epsilon = 1e-5
loss_plus = forward_and_loss(w1 + epsilon, b1, w2, b2)[3]
loss_minus = forward_and_loss(w1 - epsilon, b1, w2, b2)[3]
finite_difference = (loss_plus - loss_minus) / (2 * epsilon)
print('analytic dL/dw1:', d_loss_d_w1, 'finite-difference:', finite_difference)
assert abs(finite_difference - d_loss_d_w1) < 1e-8